# Notebook 02: Rigorous Predictive Modeling of Wage Determinants

**EquiPay Canada — Research-Grade Machine Learning Analysis**

---

## Overview

This notebook develops and evaluates predictive models for hourly wages using the Labour Force Survey (LFS) PUMF data (2010-2025). We implement a **rigorous model development pipeline** following best practices in applied machine learning for causal inference.

### Research Questions
1. How accurately can wages be predicted from observable characteristics?
2. What is the magnitude of the gender wage gap after controlling for human capital?
3. Which model provides the best predictive performance with proper uncertainty quantification?

## Methodological Framework

### Model Development Pipeline

| Stage | Method | Purpose |
|-------|--------|---------|
| **Feature Engineering** | VIF Analysis, Correlation Screening | Multicollinearity diagnosis |
| **Model Selection** | Nested Cross-Validation | Unbiased performance estimation |
| **Hyperparameter Tuning** | Grid/Random Search with CV | Optimize model complexity |
| **Model Diagnostics** | Residual Analysis, Calibration | Validate assumptions |
| **Uncertainty Quantification** | Bootstrap CIs, Prediction Intervals | Honest error bars |
| **Interpretability** | SHAP Values, Permutation Importance | Feature attribution |

### Model Comparison Framework

| Model | Type | Strengths | Assumptions |
|-------|------|-----------|-------------|
| **OLS** | Parametric | Interpretable coefficients | Linearity, normality, homoscedasticity |
| **Ridge/Lasso** | Regularized | Prevents overfitting | Same as OLS |
| **Random Forest** | Ensemble | Captures non-linearity | None (non-parametric) |
| **XGBoost** | Gradient Boosting | SOTA performance | None (non-parametric) |

### Nested Cross-Validation

**Why nested CV?** (Cawley & Talbot, 2010)
- **Inner loop**: Hyperparameter tuning
- **Outer loop**: Unbiased performance estimation
- Prevents information leakage and optimistic bias

```
Outer Fold 1: [Test] | [Inner CV for tuning ────────────]
Outer Fold 2: ─────── [Test] | [Inner CV ─────────────────]
Outer Fold 3: ──────────────── [Test] | [Inner CV ────────]
...
```

### Key Econometric Considerations
- **Log-linear specification**: $\ln(W_i) = X\beta + \gamma \cdot \text{Female}_i + \epsilon_i$ (Mincer, 1974)
- **Robust inference**: HC3 standard errors for heteroskedasticity (MacKinnon & White, 1985)
- **Effect sizes**: Report with confidence intervals (APA 7th Edition)

## Data Source
- **Labour Force Survey PUMF** (Statistics Canada Catalogue 71M0001X)
- **Period**: 2010-2025 (16 years, N ≈ 100,000+ observations)
- **Sample**: Working-age population with valid hourly wages

---

In [ ]:
# ============================================================================
# SETUP: Import Libraries and Configure Environment
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# Statistical tests
from scipy import stats as sp_stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

from src.constants import COLS, GENDER_CODES, normalize_column_names, DATA_SCOPE_START, DATA_SCOPE_END

# Publication-quality figures
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white'
})
plt.style.use('seaborn-v0_8-whitegrid')

# Ensure figures directory exists
Path('../reports/figures').mkdir(parents=True, exist_ok=True)

print("✓ Libraries loaded successfully")
print(f"✓ Analysis period: {DATA_SCOPE_START}-{DATA_SCOPE_END}")

## 1. Data Loading & Preprocessing

Load the processed LFS data prepared in Notebook 01. We apply:
- Column normalization
- Creation of derived features
- Target variable transformation

In [ ]:
# Load processed data
data_path = Path('../data/processed/lfs_processed.csv')

if not data_path.exists():
    print("⚠️ Run notebook 01 first to generate data!")
    raise FileNotFoundError(f"{data_path} not found")

df = pd.read_csv(data_path)
df = normalize_column_names(df)

print(f"✓ Loaded {len(df):,} records")
print(f"✓ Columns: {list(df.columns)}")

In [ ]:
# Identify columns
gender_col = COLS.GENDER if COLS.GENDER in df.columns else 'SEX'

# Use REAL hourly earnings for cross-year model training (inflation-adjusted)
# This ensures predictions are in 2010 constant dollars
if COLS.REAL_HOURLY_EARNINGS in df.columns:
    wage_col = COLS.REAL_HOURLY_EARNINGS
    print("✓ Using REAL hourly earnings (inflation-adjusted to 2010$)")
elif 'REAL_HRLYEARN' in df.columns:
    wage_col = 'REAL_HRLYEARN'
    print("✓ Using REAL hourly earnings (inflation-adjusted to 2010$)")
else:
    wage_col = COLS.HOURLY_EARNINGS if COLS.HOURLY_EARNINGS in df.columns else 'HRLYEARN'
    print("⚠ Real wages not available - using nominal wages")

print(f"Gender column: {gender_col}")
print(f"Wage column: {wage_col}")

# Create IS_FEMALE
df['IS_FEMALE'] = (df[gender_col] == GENDER_CODES.get('Female', 2)).astype(int)

## 2. Feature Engineering & Multicollinearity Assessment

**Critical diagnostic step:** We assess multicollinearity using the Variance Inflation Factor (VIF). 

**VIF Interpretation:**
- VIF = 1: No correlation with other predictors
- VIF < 5: Acceptable
- VIF > 10: Severe multicollinearity (requires remediation)

In [ ]:
# Define features
potential_features = [
    'IS_FEMALE', COLS.EDUC, COLS.NOC_10, COLS.PROV, 
    COLS.AGE_12, COLS.TENURE, COLS.FTPTMAIN, COLS.UNION,
    'EXPERIENCE', 'EXPERIENCE_SQ'
]

# Filter to available features
features = [f for f in potential_features if f in df.columns]
print(f"Available features: {features}")

# Target
target = wage_col

# Remove rows with missing target
df_model = df[features + [target]].dropna()
print(f"\nRows for modeling: {len(df_model):,}")

In [ ]:
# Prepare X and y
X = df_model[features]
y = df_model[target]

# Log transform target for better distribution
y_log = np.log(y.clip(lower=1))

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nTarget statistics:")
print(f"  Original - Mean: ${y.mean():.2f}, Std: ${y.std():.2f}")
print(f"  Log - Mean: {y_log.mean():.3f}, Std: {y_log.std():.3f}")

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")

In [ ]:
# Variance Inflation Factor (VIF) Analysis
print("=" * 60)
print("MULTICOLLINEARITY DIAGNOSTICS: VARIANCE INFLATION FACTOR")
print("=" * 60)

# Calculate VIF for each feature
X_vif = X.dropna()
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data.sort_values("VIF", ascending=False)

print(f"\n{'Feature':<20} {'VIF':>10} {'Status':>15}")
print("-" * 50)
for _, row in vif_data.iterrows():
    if row['VIF'] > 10:
        status = "⚠️ SEVERE"
    elif row['VIF'] > 5:
        status = "⚠ MODERATE"
    else:
        status = "✓ OK"
    print(f"{row['Feature']:<20} {row['VIF']:>10.2f} {status:>15}")

# Summary
high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\n⚠️ WARNING: {len(high_vif)} feature(s) with VIF > 10")
    print("   Consider removing or combining these features.")
else:
    print("\n✓ No severe multicollinearity detected (all VIF < 10)")

## 3. Baseline Model: OLS Regression

**Mincer wage equation** (Mincer, 1974):
$$\ln(W_i) = \beta_0 + \beta_1 S_i + \beta_2 X_i + \beta_3 X_i^2 + \gamma \text{Female}_i + \epsilon_i$$

Where:
- $S_i$ = Years of schooling (proxied by education level)
- $X_i$ = Experience
- $\gamma$ = Gender wage gap coefficient (our primary interest)

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)

# Metrics
print("LINEAR REGRESSION RESULTS")
print("=" * 40)
print(f"R² Score: {r2_score(y_test, y_pred_lr):.4f}")
print(f"RMSE (log): {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}")
print(f"MAE (log): {mean_absolute_error(y_test, y_pred_lr):.4f}")

# Convert back to dollars for interpretation
y_test_dollars = np.exp(y_test)
y_pred_dollars = np.exp(y_pred_lr)
print(f"\nIn dollars:")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars)):.2f}")
print(f"MAE: ${mean_absolute_error(y_test_dollars, y_pred_dollars):.2f}")

In [ ]:
# Feature coefficients
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nFeature Coefficients (Linear Regression):")
print("-" * 50)
for _, row in coef_df.iterrows():
    pct_effect = (np.exp(row['Coefficient']) - 1) * 100
    print(f"{row['Feature']:<20} {row['Coefficient']:>10.4f}  ({pct_effect:>+6.1f}%)")

# Gender effect
gender_coef = coef_df[coef_df['Feature'] == 'IS_FEMALE']['Coefficient'].values[0]
gender_effect = (np.exp(gender_coef) - 1) * 100
print(f"\n*** Gender Effect: {gender_effect:.1f}% (negative = women earn less) ***")

In [ ]:
# Residual Diagnostics for OLS
residuals = y_test - y_pred_lr

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(y_pred_lr, residuals, alpha=0.3, s=10)
ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Fitted Values (log wage)')
ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted', fontweight='bold')

# 2. Q-Q Plot of Residuals
ax = axes[0, 1]
sp_stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title('Q-Q Plot of Residuals', fontweight='bold')

# 3. Histogram of Residuals
ax = axes[1, 0]
ax.hist(residuals, bins=50, edgecolor='white', alpha=0.7, density=True)
xmin, xmax = ax.get_xlim()
x = np.linspace(xmin, xmax, 100)
p = sp_stats.norm.pdf(x, residuals.mean(), residuals.std())
ax.plot(x, p, 'r-', linewidth=2, label='Normal Distribution')
ax.set_xlabel('Residuals')
ax.set_ylabel('Density')
ax.set_title('Distribution of Residuals', fontweight='bold')
ax.legend()

# 4. Scale-Location Plot
ax = axes[1, 1]
ax.scatter(y_pred_lr, np.sqrt(np.abs(residuals)), alpha=0.3, s=10)
ax.set_xlabel('Fitted Values')
ax.set_ylabel('√|Residuals|')
ax.set_title('Scale-Location (Homoscedasticity Check)', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/ols_residual_diagnostics.png', dpi=150, 
            bbox_inches='tight', facecolor='white')
plt.show()

# Formal tests
print("\n" + "=" * 60)
print("RESIDUAL DIAGNOSTIC TESTS")
print("=" * 60)

# Jarque-Bera test for normality
jb_stat, jb_pval = sp_stats.jarque_bera(residuals)
print(f"\nJarque-Bera Test (Normality):")
print(f"  Statistic: {jb_stat:.2f}")
print(f"  p-value: {jb_pval:.4e}")
print(f"  Conclusion: {'Residuals NOT normal' if jb_pval < 0.05 else 'Residuals normal'}")

# Durbin-Watson (approximate using lag-1 autocorrelation)
from statsmodels.stats.stattools import durbin_watson
dw_stat = durbin_watson(residuals)
print(f"\nDurbin-Watson Test (Autocorrelation):")
print(f"  Statistic: {dw_stat:.4f}")
print(f"  Interpretation: ~2 = no autocorrelation, <1.5 = positive, >2.5 = negative")

print("\n📊 Figure saved: reports/figures/ols_residual_diagnostics.png")

## 4. Regularized Models: Ridge & Lasso Regression

**Purpose of regularization:**
- **Ridge** (L2): Shrinks coefficients toward zero; retains all features
- **Lasso** (L1): Performs feature selection; sets some coefficients exactly to zero

In [ ]:
# Ridge Regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
y_pred_ridge = ridge_model.predict(X_test_scaled)

print("RIDGE REGRESSION RESULTS")
print("=" * 40)
print(f"R² Score: {r2_score(y_test, y_pred_ridge):.4f}")
print(f"RMSE (log): {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.4f}")

In [ ]:
# Lasso Regression
lasso_model = Lasso(alpha=0.001)
lasso_model.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_model.predict(X_test_scaled)

print("LASSO REGRESSION RESULTS")
print("=" * 40)
print(f"R² Score: {r2_score(y_test, y_pred_lasso):.4f}")
print(f"RMSE (log): {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.4f}")

# Show selected features (non-zero coefficients)
selected = [(f, c) for f, c in zip(features, lasso_model.coef_) if abs(c) > 0.001]
print(f"\nSelected features: {len(selected)}/{len(features)}")

## 5. Ensemble Methods: Random Forest

**Advantages:**
- Captures non-linear relationships
- Handles feature interactions automatically
- Provides feature importance measures
- Robust to outliers

In [ ]:
# Random Forest (using unscaled features)
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("RANDOM FOREST RESULTS")
print("=" * 40)
print(f"R² Score: {r2_score(y_test, y_pred_rf):.4f}")
print(f"RMSE (log): {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}")

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if f == 'IS_FEMALE' else '#3498db' for f in importance_df['Feature']]
ax.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importance\n(Red = Gender)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Gradient Boosting & XGBoost

**Sequential ensemble learning:** Each tree corrects errors from previous trees, achieving state-of-the-art predictive performance.

In [ ]:
# Gradient Boosting
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

print("GRADIENT BOOSTING RESULTS")
print("=" * 40)
print(f"R² Score: {r2_score(y_test, y_pred_gb):.4f}")
print(f"RMSE (log): {np.sqrt(mean_squared_error(y_test, y_pred_gb)):.4f}")

In [ ]:
# Try XGBoost if available
try:
    import xgboost as xgb
    
    xgb_model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    
    print("XGBOOST RESULTS")
    print("=" * 40)
    print(f"R² Score: {r2_score(y_test, y_pred_xgb):.4f}")
    print(f"RMSE (log): {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.4f}")
    HAS_XGB = True
except ImportError:
    print("XGBoost not installed. Skipping...")
    HAS_XGB = False
    y_pred_xgb = y_pred_gb  # fallback

## 7. Model Comparison & Evaluation

**Metrics:**
- **R²**: Proportion of variance explained (higher = better)
- **RMSE**: Root Mean Square Error (lower = better)
- **MAE**: Mean Absolute Error (lower = better, more robust to outliers)

In [ ]:
# Compare all models
models = {
    'Linear Regression': y_pred_lr,
    'Ridge': y_pred_ridge,
    'Lasso': y_pred_lasso,
    'Random Forest': y_pred_rf,
    'Gradient Boosting': y_pred_gb,
}

if HAS_XGB:
    models['XGBoost'] = y_pred_xgb

results = []
for name, preds in models.items():
    results.append({
        'Model': name,
        'R²': r2_score(y_test, preds),
        'RMSE (log)': np.sqrt(mean_squared_error(y_test, preds)),
        'MAE (log)': mean_absolute_error(y_test, preds),
        'RMSE ($)': np.sqrt(mean_squared_error(np.exp(y_test), np.exp(preds))),
        'MAE ($)': mean_absolute_error(np.exp(y_test), np.exp(preds))
    })

results_df = pd.DataFrame(results).sort_values('R²', ascending=False)
print("MODEL COMPARISON")
print("=" * 80)
display(results_df.round(4))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² comparison
ax = axes[0]
bars = ax.barh(results_df['Model'], results_df['R²'], color='steelblue')
ax.set_xlabel('R² Score')
ax.set_title('Model R² Comparison')
ax.set_xlim(0, 1)

# RMSE comparison
ax = axes[1]
bars = ax.barh(results_df['Model'], results_df['RMSE ($)'], color='coral')
ax.set_xlabel('RMSE ($)')
ax.set_title('Model RMSE Comparison (lower is better)')

plt.tight_layout()
plt.show()

## 8. Gender Effect Analysis Across Models

**Key research question:** After controlling for observable characteristics, how much of the wage gap remains attributable to gender?

This "unexplained" gap may reflect:
- Discrimination
- Omitted variables (e.g., actual work experience, hours worked)
- Occupational choice endogeneity

In [ ]:
# Extract gender effect from each model
print("GENDER EFFECT BY MODEL")
print("=" * 60)
print("(Negative = women earn less than men, controlling for other factors)")
print()

# Linear models have explicit coefficients
gender_idx = features.index('IS_FEMALE')

print(f"{'Model':<25} {'Coefficient':>12} {'Effect (%)':>12}")
print("-" * 50)

lr_gender = lr_model.coef_[gender_idx]
print(f"{'Linear Regression':<25} {lr_gender:>12.4f} {(np.exp(lr_gender)-1)*100:>11.1f}%")

ridge_gender = ridge_model.coef_[gender_idx]
print(f"{'Ridge':<25} {ridge_gender:>12.4f} {(np.exp(ridge_gender)-1)*100:>11.1f}%")

lasso_gender = lasso_model.coef_[gender_idx]
print(f"{'Lasso':<25} {lasso_gender:>12.4f} {(np.exp(lasso_gender)-1)*100:>11.1f}%")

# For tree models, calculate average effect
X_male = X_test.copy()
X_male['IS_FEMALE'] = 0
X_female = X_test.copy()
X_female['IS_FEMALE'] = 1

rf_effect = rf_model.predict(X_female).mean() - rf_model.predict(X_male).mean()
print(f"{'Random Forest':<25} {rf_effect:>12.4f} {(np.exp(rf_effect)-1)*100:>11.1f}%")

gb_effect = gb_model.predict(X_female).mean() - gb_model.predict(X_male).mean()
print(f"{'Gradient Boosting':<25} {gb_effect:>12.4f} {(np.exp(gb_effect)-1)*100:>11.1f}%")

if HAS_XGB:
    xgb_effect = xgb_model.predict(X_female).mean() - xgb_model.predict(X_male).mean()
    print(f"{'XGBoost':<25} {xgb_effect:>12.4f} {(np.exp(xgb_effect)-1)*100:>11.1f}%")

## 9.2 Nested Cross-Validation for Unbiased Estimation

**Why Nested CV?** (Cawley & Talbot, 2010; Varma & Simon, 2006)

Standard cross-validation is **optimistically biased** when hyperparameters are tuned. Nested CV provides:
- Unbiased generalization error estimate
- No information leakage between tuning and evaluation
- Proper uncertainty quantification

| Structure | Purpose |
|-----------|---------|
| **Outer loop** (5-fold) | Estimate generalization error |
| **Inner loop** (3-fold) | Hyperparameter tuning |

In [ ]:
# ============================================================================
# NESTED CROSS-VALIDATION FOR UNBIASED PERFORMANCE ESTIMATION
# ============================================================================
# Implements the double CV procedure of Cawley & Talbot (2010)
# - Outer loop: 5-fold CV for unbiased performance estimation
# - Inner loop: 3-fold CV for hyperparameter tuning
# ============================================================================

from sklearn.model_selection import cross_val_score, KFold, GridSearchCV

print("=" * 70)
print("NESTED CROSS-VALIDATION (Unbiased Performance Estimation)")
print("=" * 70)

# Define models with hyperparameter grids
nested_cv_models = {
    'Ridge': {
        'estimator': Ridge(),
        'param_grid': {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
    },
    'Random Forest': {
        'estimator': RandomForestRegressor(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [50, 100],
            'max_depth': [5, 10, 15],
            'min_samples_leaf': [5, 10]
        }
    }
}

# Outer CV loop
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

nested_results = []

for name, config in nested_cv_models.items():
    print(f"\nEvaluating {name}...")
    
    # Inner CV for hyperparameter tuning
    grid_search = GridSearchCV(
        config['estimator'],
        config['param_grid'],
        cv=inner_cv,
        scoring='r2',
        n_jobs=-1
    )
    
    # Outer CV for unbiased performance
    # Use scaled data for linear models
    if 'Ridge' in name or 'Linear' in name:
        X_scaled = scaler.fit_transform(X)
        outer_scores = cross_val_score(grid_search, X_scaled, y_log, cv=outer_cv, scoring='r2')
    else:
        outer_scores = cross_val_score(grid_search, X, y_log, cv=outer_cv, scoring='r2')
    
    nested_results.append({
        'Model': name,
        'Nested CV Mean R²': outer_scores.mean(),
        'Nested CV Std': outer_scores.std(),
        'CV Scores': outer_scores
    })
    
    print(f"  Nested CV R²: {outer_scores.mean():.4f} ± {outer_scores.std():.4f}")
    print(f"  Fold scores: {', '.join([f'{s:.4f}' for s in outer_scores])}")

# Compare nested CV results
print("\n" + "=" * 70)
print("NESTED CV RESULTS SUMMARY")
print("=" * 70)
print(f"\n{'Model':<20} {'Nested CV R²':>18} {'Std Dev':>12}")
print("-" * 50)

for nested_result in nested_results:
    model_name = nested_result['Model']
    nested_r2 = nested_result['Nested CV Mean R²']
    nested_std = nested_result['Nested CV Std']
    print(f"{model_name:<20} {nested_r2:>18.4f} {nested_std:>12.4f}")

print("\n📊 Note: Nested CV provides unbiased performance estimates")

## 9.3 Learning Curves: Bias-Variance Diagnosis

**Learning curves** plot training and validation scores against training set size to diagnose:
- **High Bias** (underfitting): Both scores converge at low value → need more complex model
- **High Variance** (overfitting): Large gap between scores → need more data or regularization
- **Good Fit**: Both scores converge at high value with small gap

In [ ]:
# ============================================================================
# LEARNING CURVES FOR BIAS-VARIANCE DIAGNOSIS
# ============================================================================

from sklearn.model_selection import learning_curve

print("=" * 70)
print("LEARNING CURVE ANALYSIS (Bias-Variance Trade-off)")
print("=" * 70)

def plot_learning_curves(estimator, title, X, y, cv=5, scoring='r2'):
    """
    Generate learning curve plot for bias-variance diagnosis.
    """
    # Limit sample sizes for computational efficiency
    n_samples = len(X)
    train_sizes = np.linspace(0.1, 1.0, 10)
    
    train_sizes_abs, train_scores, val_scores = learning_curve(
        estimator, X, y,
        train_sizes=train_sizes,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        random_state=42
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    return train_sizes_abs, train_mean, train_std, val_mean, val_std

# Sample data for faster computation
sample_size = min(10000, len(X))
X_sample = X.sample(sample_size, random_state=42)
y_sample = y_log.loc[X_sample.index]
X_sample_scaled = scaler.fit_transform(X_sample)

# Learning curves for different models
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

models_for_lc = [
    ('Linear Regression', LinearRegression(), X_sample_scaled, axes[0, 0]),
    ('Ridge (α=1.0)', Ridge(alpha=1.0), X_sample_scaled, axes[0, 1]),
    ('Random Forest', RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42), X_sample, axes[1, 0]),
    ('Gradient Boosting', GradientBoostingRegressor(n_estimators=50, max_depth=5, random_state=42), X_sample, axes[1, 1])
]

for name, model, X_lc, ax in models_for_lc:
    train_sizes, train_mean, train_std, val_mean, val_std = plot_learning_curves(
        model, name, X_lc, y_sample, cv=5
    )
    
    # Plot training scores
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Training Score')
    
    # Plot validation scores
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
    ax.plot(train_sizes, val_mean, 'o-', color='orange', label='Cross-Validation Score')
    
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('R² Score')
    ax.set_title(f'{name}', fontweight='bold')
    ax.legend(loc='lower right')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    
    # Diagnose bias-variance
    final_gap = train_mean[-1] - val_mean[-1]
    if final_gap > 0.1:
        diagnosis = "High Variance (overfitting)"
    elif val_mean[-1] < 0.3:
        diagnosis = "High Bias (underfitting)"
    else:
        diagnosis = "Balanced"
    ax.annotate(diagnosis, xy=(0.5, 0.05), xycoords='axes fraction', 
                fontsize=9, ha='center', style='italic')

plt.tight_layout()
plt.savefig('../reports/figures/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Learning curves saved to reports/figures/learning_curves.png")

## 9.4 SHAP Values: Model-Agnostic Interpretability

**SHAP (SHapley Additive exPlanations)** (Lundberg & Lee, 2017) provides:
- Consistent, locally accurate feature attributions
- Based on game-theoretic Shapley values
- Applicable to any model (not just tree-based)

| Interpretation | Meaning |
|----------------|---------|
| **Positive SHAP** | Feature pushes prediction higher |
| **Negative SHAP** | Feature pushes prediction lower |
| **Magnitude** | Importance of feature for that prediction |

In [ ]:
# ============================================================================
# SHAP VALUES FOR MODEL INTERPRETABILITY
# ============================================================================
# Lundberg & Lee (2017): "A Unified Approach to Interpreting Model Predictions"
# ============================================================================

try:
    import shap
    
    print("=" * 70)
    print("SHAP VALUE ANALYSIS (Model-Agnostic Interpretability)")
    print("=" * 70)
    
    # Use a sample for SHAP computation (SHAP can be slow for large datasets)
    shap_sample_size = min(500, len(X_test))
    X_shap = X_test.sample(shap_sample_size, random_state=42)
    
    # Use TreeExplainer for tree-based models (fast)
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_shap)
    
    # Summary plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Beeswarm plot
    plt.subplot(1, 2, 1)
    shap.summary_plot(shap_values, X_shap, feature_names=features, 
                      plot_type="violin", show=False)
    plt.title('SHAP Value Distribution by Feature', fontweight='bold')
    
    # Bar plot (mean absolute SHAP values)
    plt.subplot(1, 2, 2)
    shap.summary_plot(shap_values, X_shap, feature_names=features, 
                      plot_type="bar", show=False)
    plt.title('Mean |SHAP Value| (Feature Importance)', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../reports/figures/shap_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # SHAP values for gender specifically
    gender_idx = features.index('IS_FEMALE')
    gender_shap = shap_values[:, gender_idx]
    
    print(f"\nGender (IS_FEMALE) SHAP Analysis:")
    print(f"  Mean SHAP value:   {gender_shap.mean():.4f}")
    print(f"  Median SHAP value: {np.median(gender_shap):.4f}")
    print(f"  Std SHAP value:    {gender_shap.std():.4f}")
    print(f"  Effect range:      [{gender_shap.min():.4f}, {gender_shap.max():.4f}]")
    
    # Interpretation
    avg_gender_effect = gender_shap.mean()
    pct_effect = (np.exp(avg_gender_effect) - 1) * 100
    print(f"\n  → Average gender effect: {pct_effect:+.2f}% on wages")
    print(f"    (negative = women earn less)")
    
    print("\n📊 SHAP plots saved to reports/figures/shap_analysis.png")
    
except ImportError:
    print("⚠️ SHAP not installed. Install with: pip install shap")
    print("   Skipping SHAP analysis...")

## 9.5 Prediction Intervals: Uncertainty Quantification

**Point predictions are not enough** — we need uncertainty quantification:
- **Confidence intervals**: Uncertainty in the mean prediction
- **Prediction intervals**: Uncertainty for a new individual observation

For Random Forest, we use **quantile regression forests** or bootstrap aggregation to estimate prediction intervals.

In [ ]:
# ============================================================================
# PREDICTION INTERVALS VIA BOOTSTRAP
# ============================================================================
# Following Efron & Tibshirani (1993) bootstrap methodology
# ============================================================================

print("=" * 70)
print("PREDICTION INTERVALS VIA BOOTSTRAP")
print("=" * 70)

def bootstrap_prediction_intervals(model, X_train, y_train, X_test, 
                                   n_bootstraps=100, alpha=0.05):
    """
    Compute prediction intervals using bootstrap aggregation.
    
    Parameters
    ----------
    model : sklearn estimator
        Base model to bootstrap
    X_train, y_train : training data
    X_test : test features
    n_bootstraps : int
        Number of bootstrap samples
    alpha : float
        Significance level (0.05 for 95% CI)
    
    Returns
    -------
    predictions, lower_bound, upper_bound
    """
    n_train = len(X_train)
    n_test = len(X_test)
    
    # Store predictions from each bootstrap
    bootstrap_preds = np.zeros((n_bootstraps, n_test))
    
    for i in range(n_bootstraps):
        # Sample with replacement
        idx = np.random.choice(n_train, size=n_train, replace=True)
        X_boot = X_train.iloc[idx] if hasattr(X_train, 'iloc') else X_train[idx]
        y_boot = y_train.iloc[idx] if hasattr(y_train, 'iloc') else y_train[idx]
        
        # Fit model and predict
        model_clone = type(model)(**model.get_params())
        model_clone.fit(X_boot, y_boot)
        bootstrap_preds[i, :] = model_clone.predict(X_test)
    
    # Calculate prediction intervals
    predictions = bootstrap_preds.mean(axis=0)
    lower = np.percentile(bootstrap_preds, (alpha/2) * 100, axis=0)
    upper = np.percentile(bootstrap_preds, (1 - alpha/2) * 100, axis=0)
    
    return predictions, lower, upper

# Compute prediction intervals for a subset (bootstrap is expensive)
pi_sample_size = min(500, len(X_test))
X_test_pi = X_test.sample(pi_sample_size, random_state=42)
y_test_pi = y_test.loc[X_test_pi.index]

print(f"Computing 95% prediction intervals via {50} bootstrap samples...")

preds, lower, upper = bootstrap_prediction_intervals(
    RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1),
    X_train, y_train, X_test_pi, n_bootstraps=50
)

# Coverage calculation
coverage = np.mean((y_test_pi >= lower) & (y_test_pi <= upper))
avg_width = np.mean(upper - lower)

print(f"\n95% Prediction Interval Results:")
print(f"  Coverage: {coverage*100:.1f}% (target: 95%)")
print(f"  Average interval width: {avg_width:.3f} (log scale)")
print(f"  Average width in $: ${np.exp(avg_width):.2f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sort by predicted value for cleaner plot
sort_idx = np.argsort(preds)[:100]  # First 100 for visibility

ax = axes[0]
ax.fill_between(range(len(sort_idx)), 
                np.exp(lower[sort_idx]), np.exp(upper[sort_idx]), 
                alpha=0.3, label='95% PI')
ax.scatter(range(len(sort_idx)), np.exp(y_test_pi.iloc[sort_idx]), 
           c='red', s=20, alpha=0.7, label='Actual')
ax.plot(range(len(sort_idx)), np.exp(preds[sort_idx]), 'b-', 
        linewidth=1.5, label='Predicted')
ax.set_xlabel('Observation (sorted by prediction)')
ax.set_ylabel('Hourly Wage ($)')
ax.set_title('Prediction Intervals (95% Bootstrap)')
ax.legend()

# Coverage by prediction level
ax = axes[1]
n_bins = 5
bins = np.percentile(preds, np.linspace(0, 100, n_bins + 1))
bin_coverage = []
for i in range(n_bins):
    mask = (preds >= bins[i]) & (preds < bins[i+1])
    if mask.sum() > 0:
        bin_cov = np.mean((y_test_pi.values[mask] >= lower[mask]) & 
                          (y_test_pi.values[mask] <= upper[mask]))
        bin_coverage.append(bin_cov * 100)
    else:
        bin_coverage.append(0)

ax.bar(range(n_bins), bin_coverage, color='steelblue', edgecolor='black')
ax.axhline(95, color='red', linestyle='--', label='Target 95%')
ax.set_xlabel('Prediction Quintile')
ax.set_ylabel('Coverage (%)')
ax.set_title('Coverage by Prediction Level')
ax.set_xticks(range(n_bins))
ax.set_xticklabels([f'Q{i+1}' for i in range(n_bins)])
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/prediction_intervals.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Prediction intervals saved to reports/figures/prediction_intervals.png")

In [ ]:
# Cross-validation for best model
print("5-FOLD CROSS-VALIDATION")
print("=" * 40)

cv_models = [
    ('Linear Regression', LinearRegression()),
    ('Ridge', Ridge(alpha=1.0)),
    ('Random Forest', RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)),
    ('Gradient Boosting', GradientBoostingRegressor(n_estimators=50, max_depth=5, random_state=42))
]

cv_results = []
for name, model in cv_models:
    # Use scaled data for linear models
    if 'Linear' in name or 'Ridge' in name:
        scores = cross_val_score(model, scaler.fit_transform(X), y_log, cv=5, scoring='r2')
    else:
        scores = cross_val_score(model, X, y_log, cv=5, scoring='r2')
    
    cv_results.append({
        'Model': name,
        'Mean R²': scores.mean(),
        'Std': scores.std(),
        'Min': scores.min(),
        'Max': scores.max()
    })
    print(f"{name}: R² = {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

cv_df = pd.DataFrame(cv_results)
display(cv_df.round(4))

## 10. Model Export & Conclusions

In [ ]:
# Select best model based on R²
best_model_name = results_df.iloc[0]['Model']
print(f"Best model: {best_model_name}")

# Save the models
models_path = Path('../models')
models_path.mkdir(exist_ok=True)

# Save the best performing model
if 'Random Forest' in best_model_name:
    best_model = rf_model
elif 'XGBoost' in best_model_name and HAS_XGB:
    best_model = xgb_model
elif 'Gradient' in best_model_name:
    best_model = gb_model
else:
    best_model = lr_model

joblib.dump(best_model, models_path / 'salary_predictor.joblib')
joblib.dump(scaler, models_path / 'feature_scaler.joblib')

print(f"\n✓ Saved model to {models_path / 'salary_predictor.joblib'}")
print(f"✓ Saved scaler to {models_path / 'feature_scaler.joblib'}")

In [ ]:
# Summary
print("\n" + "=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)

print(f"\nDataset: {len(df):,} records")
print(f"Features: {len(features)}")
print(f"Best model: {best_model_name}")
print(f"Best R²: {results_df.iloc[0]['R²']:.4f}")
print(f"Best RMSE: ${results_df.iloc[0]['RMSE ($)']:.2f}")

print(f"\nKey Finding:")
print(f"Controlling for education, occupation, experience, and other factors,")
print(f"women earn approximately {abs((np.exp(lr_gender)-1)*100):.1f}% less than men.")

---

## References

### Data Source
- Statistics Canada. (2024). Labour Force Survey Public Use Microdata File (Catalogue 71M0001X).

### Economic Theory
- Mincer, J. (1974). *Schooling, Experience, and Earnings*. NBER.

### Machine Learning Methodology
- Cawley, G. C., & Talbot, N. L. (2010). On over-fitting in model selection and subsequent selection bias. *JMLR*, 11, 2079-2107.
- Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
- Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *Annals of Statistics*, 29(5), 1189-1232.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer.
- Varma, S., & Simon, R. (2006). Bias in error estimation when using cross-validation. *BMC Bioinformatics*, 7, 91.

### Interpretability
- Lundberg, S. M., & Lee, S. I. (2017). A unified approach to interpreting model predictions. *NeurIPS*, 30.

### Robust Inference
- MacKinnon, J. G., & White, H. (1985). Some heteroskedasticity-consistent covariance matrix estimators. *Econometrica*, 53(3), 817-838.

---

## Limitations

1. **Omitted Variable Bias**: Actual work experience, career interruptions not observed in LFS PUMF
2. **Endogeneity**: Occupation choice may be endogenous to gender
3. **Selection Bias**: Only wage earners observed (see Heckman correction in NB05)
4. **Causal Interpretation**: ML models estimate predictive, not causal, relationships

---

**Next:** [03_pay_equity_analysis.ipynb](03_pay_equity_analysis.ipynb) - Blinder-Oaxaca Decomposition of the Gender Wage Gap